# 1. Molecules

The `SMolecule` class is genepie's in-memory representation of a molecular
structure — the Python mirror of GENESIS's Fortran `s_molecule` type. It holds
per-atom arrays (names, residues, masses, coordinates) plus the bonded topology,
and it is the first thing every analysis needs.


In [ ]:
import numpy as np
from genepie import genesis_exe, SMolecule
from genepie.tests.conftest import BPTI_PDB, BPTI_PSF

mol = SMolecule.from_file(pdb=BPTI_PDB, psf=BPTI_PSF, ref=BPTI_PDB)
print(f"num_atoms    = {mol.num_atoms}")
print(f"num_residues = {mol.num_residues}")
print(f"num_bonds    = {mol.num_bonds}")
print(f"total_charge = {mol.total_charge:.3f}")

## Per-atom arrays

The fields are plain NumPy arrays, so slicing and boolean masks work as usual.

In [ ]:
print("first 5 atom names   :", mol.atom_name[:5])
print("first 5 residue names:", mol.residue_name[:5])
print("first 5 residue nos  :", mol.residue_no[:5])
print("atom_coord shape      :", mol.atom_coord.shape, "(natom, 3)")
print("mass shape            :", mol.mass.shape)

## Selecting atoms

`genesis_exe.selection` evaluates a GENESIS selection expression and returns the
matching atom indices (1-indexed, following the Fortran convention). Selection
strings are the same ones you pass to the analysis functions.


In [ ]:
ca = genesis_exe.selection(mol, "an:CA")
print(f"{len(ca)} C-alpha atoms; first indices: {ca[:5]}")

protein = genesis_exe.selection(mol, "sid:BPTI")
print(f"{len(protein)} atoms in segment BPTI (out of {mol.num_atoms} total)")

## Visualize the structure

Because the fields are just arrays, they interoperate with the wider ecosystem.
Here we render the protein with [py3Dmol](https://pypi.org/project/py3Dmol/).
Waters and ions are filtered out so the cartoon is easy to read.


In [ ]:
import py3Dmol

skip = {"TIP3", "WAT", "HOH", "SOD", "CLA", "POT", "NA", "CL"}
protein_pdb = "".join(
    line for line in open(BPTI_PDB)
    if line.startswith(("ATOM", "HETATM")) and line[17:20].strip() not in skip
)

view = py3Dmol.view(width=600, height=420)
view.addModel(protein_pdb, "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()

# Emit the viewer as self-contained HTML so it embeds in the static site
# (the default py3Dmol mime type only renders inside a live Jupyter session).
from IPython.display import HTML
HTML(view._make_html())

## Integration with MDTraj / MDAnalysis

`SMolecule` converts to and from both toolkits, so you can reuse their selection
languages, readers, and writers:

- `mol.to_mdtraj_topology()` / `SMolecule.from_mdtraj_topology(top)`
- `mol.to_mdanalysis_universe()` / `SMolecule.from_mdanalysis_universe(u)`

The [ML integration chapter](08_ml_integration.ipynb) uses these bridges to feed
GENESIS results into scikit-learn and PyTorch.
